In [121]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [67]:
true = pd.read_csv("True.csv")
true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [68]:
fake = pd.read_csv("Fake.csv")
fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [69]:
true.shape , fake.shape

((21417, 4), (23481, 4))

In [70]:
true.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21417 entries, 0 to 21416
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    21417 non-null  object
 1   text     21417 non-null  object
 2   subject  21417 non-null  object
 3   date     21417 non-null  object
dtypes: object(4)
memory usage: 669.4+ KB


In [71]:
fake.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23481 entries, 0 to 23480
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    23481 non-null  object
 1   text     23481 non-null  object
 2   subject  23481 non-null  object
 3   date     23481 non-null  object
dtypes: object(4)
memory usage: 733.9+ KB


## Data Preprocessing 

In [72]:
true["label"] = 1
fake["label"] = 0

In [73]:
true.sample(3)

,title,text,subject,date,label
10245,"Trump 'needs all the help he can get,' donors say",WASHINGTON (Reuters) - Republican presidential...,politicsNews,"March 21, 2016",1
5025,The Trump presidency on March 9 at 9:49 p.m. E...,(Reuters) - Highlights of the day for U.S. Pre...,politicsNews,"March 9, 2017",1
146,Alabama Senate race winner urges Republican ri...,WASHINGTON (Reuters) - Alabama Democrat Doug J...,politicsNews,"December 14, 2017",1


In [74]:
fake.sample(3)

,title,text,subject,date,label
7414,Bill Maher And Barney Frank Destroy GOP For C...,Bill Maher and Barney Frank totally schooled R...,News,"March 19, 2016",0
15637,UH OH! STEPHANOPOULOS JOINED CLINTON ON ‘PEDO ...,The main stream media has done a great job of ...,politics,"May 24, 2015",0
1428,It’s Happening: Justice Department Appoints S...,After the nation joined together and demanded ...,News,"May 17, 2017",0


In [75]:
# Merging the two datasets
news = pd.concat([fake,true],axis=0)
news.head(2)

,title,text,subject,date,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",0
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",0


In [76]:
news.tail(2)

,title,text,subject,date,label
21415,Vatican upbeat on possibility of Pope Francis ...,MOSCOW (Reuters) - Vatican Secretary of State ...,worldnews,"August 22, 2017",1
21416,Indonesia to buy $1.14 billion worth of Russia...,JAKARTA (Reuters) - Indonesia will buy 11 Sukh...,worldnews,"August 22, 2017",1


In [77]:
news.shape

(44898, 5)

In [78]:
news.isnull().sum()

title      0
text       0
subject    0
date       0
label      0
dtype: int64

In [79]:
news = news.drop(['title',"subject",'date'],axis=1)


In [80]:
news.sample(5)

,text,label
6891,Donald Trump is running a campaign that does n...,0
12160,"NEAR RAMALLAH, West Bank (Reuters) - I went to...",1
17407,PARIS (Reuters) - Emmanuel Macron on Sunday sa...,1
16256,OTTAWA (Reuters) - Morneau Shepell Inc (MSI.TO...,1
5523,CHICAGO (Reuters) - Nordstrom Inc said on Wedn...,1


In [81]:
# Shuffling the Dataset
news = news.sample(frac=1)

In [82]:
news.head(5)

,text,label
2140,Donald Trump s Oklahoma campaign chair is a st...,0
2094,"WASHINGTON/BRIDGEWATER, N.J. (Reuters) - Presi...",1
110,WASHINGTON (Reuters) - U.S. President Donald T...,1
6804,WASHINGTON (Reuters) - Coal mining executive R...,1
13947,PARIS (Reuters) - Barely six months into offic...,1


In [83]:
news.reset_index(inplace=True)

In [84]:
news.head(5)

,index,text,label
0,2140,Donald Trump s Oklahoma campaign chair is a st...,0
1,2094,"WASHINGTON/BRIDGEWATER, N.J. (Reuters) - Presi...",1
2,110,WASHINGTON (Reuters) - U.S. President Donald T...,1
3,6804,WASHINGTON (Reuters) - Coal mining executive R...,1
4,13947,PARIS (Reuters) - Barely six months into offic...,1


In [85]:
news = news.drop('index',axis=1)

In [86]:
news.head(5)

,text,label
0,Donald Trump s Oklahoma campaign chair is a st...,0
1,"WASHINGTON/BRIDGEWATER, N.J. (Reuters) - Presi...",1
2,WASHINGTON (Reuters) - U.S. President Donald T...,1
3,WASHINGTON (Reuters) - Coal mining executive R...,1
4,PARIS (Reuters) - Barely six months into offic...,1


In [90]:
news.shape

(44898, 2)

In [87]:
def word_operations(text):
    text = text.lower()

    text = re.sub(r'https?://\S+|www\.\S+', '', text) # Remove URLs
    text = re.sub(r'[^\w\s]', '', text) # Remove punctuation 

    text = re.sub(r'\d', '', text) # Remove digits
    text = re.sub(r'\n', ' ', text) # Remove newline characters

    text = re.sub(r'<.*?>', '', text) # Remove HTML tags

    return text

In [88]:
news['text'] = news['text'].apply(word_operations)

In [95]:
news.sample(5)

,text,label
11873,washington reuters neil gorsuch president don...,1
34689,st century wire says no it is notretired baske...,0
37357,a pair of high school students who take classe...,0
16565,berlin reuters chancellor angela merkel said ...,1
9289,reuters the tennessee house passed a bill on ...,1


## Dataset Split

In [127]:
X = news['text']
y = news['label']


In [128]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3,random_state=42)

X_train.shape , X_test.shape

((31428,), (13470,))

In [129]:
vectorizer = TfidfVectorizer(max_features=5000)

In [130]:
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

## Model Building

In [131]:
lr = LogisticRegression()
lr.fit(X_train_vec,y_train)

LogisticRegression()

In [132]:
y_pred = lr.predict(X_test_vec)

print(f"LR Accuracy Score: {accuracy_score(y_test,y_pred):.3f}")


LR Accuracy Score: 0.989


In [133]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      7126
           1       0.98      0.99      0.99      6344

    accuracy                           0.99     13470
   macro avg       0.99      0.99      0.99     13470
weighted avg       0.99      0.99      0.99     13470



In [112]:
dt = DecisionTreeClassifier()
dt.fit(X_train_vec,y_train)

DecisionTreeClassifier()

In [113]:
y_pred = dt.predict(X_test_vec)

print(f"DT Accuracy Score: {accuracy_score(y_test,y_pred):.3f}")

DT Accuracy Score: 0.995


In [114]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.99      1.00      1.00      7126
           1       0.99      0.99      0.99      6344

    accuracy                           0.99     13470
   macro avg       0.99      0.99      0.99     13470
weighted avg       0.99      0.99      0.99     13470



In [116]:
rf = RandomForestClassifier()

rf.fit(X_train_vec,y_train)

RandomForestClassifier()

In [117]:
y_pred = rf.predict(X_test_vec)

print(f"RF Accuracy Score: {accuracy_score(y_test,y_pred):.3f}")

RF Accuracy Score: 0.987


In [118]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.99      0.98      0.99      7126
           1       0.98      0.99      0.99      6344

    accuracy                           0.99     13470
   macro avg       0.99      0.99      0.99     13470
weighted avg       0.99      0.99      0.99     13470



In [122]:
gdb = GradientBoostingClassifier()
gdb.fit(X_train_vec,y_train)

GradientBoostingClassifier()

In [123]:
y_pred = gdb.predict(X_test_vec)

print(f"GD Accuracy Score: {accuracy_score(y_test,y_pred):.3f}")

GD Accuracy Score: 0.995


In [124]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       1.00      0.99      1.00      7126
           1       0.99      1.00      0.99      6344

    accuracy                           0.99     13470
   macro avg       0.99      0.99      0.99     13470
weighted avg       0.99      0.99      0.99     13470



## Saving the Model

In [134]:
import pickle

In [135]:
pickle.dump(vectorizer,open("vectorizer.pkl","wb"))
pickle.dump(lr,open("model.pkl","wb"))